In [19]:
# CONFIGURACIÓN INICIAL DEL PIPELINE DE LIMPIEZA

# Importar librerías necesarias

import pandas as pd   # Para manipulación de datos
import numpy as np    # Para operaciones numéricas
import os             # Para manejo de rutas de archivos
import json           # Para posible carga de configuración desde JSON
from pathlib import Path  # Para manejo de rutas de forma más limpia

# Definir parámetros de configuración en un diccionario
# (Esto para modificar el comportamiento del pipeline sin tocar el código)
CONFIG = {
    # Ruta del archivo de entrada
    'file_path': r'C:\Repositorios\insightengine\data\test\MuestraCredito5000V2.csv',   # Cambia según el dataset

    # Tipo de archivo: 'csv', 'excel', 'json' (por ahora)
    'file_type': 'csv',

    # Configuración específica para CSV (encoding, separador)
    'csv_encoding': 'utf-8',            # Codificación del archivo CSV
    'csv_sep': ';',','                      # Separador de columnas

    # Nombres de las columnas esenciales (deben coincidir con el dataset)
    'id_cliente': 'id_cliente',  # Columna que identifica la transacción
    'id_producto': 'id_producto',      # Columna que contiene el ítem (producto/servicio)

    'additional_filters': None,          # Diccionario con {columna: valor} para filtrar

    # Ruta de salida para el archivo de transacciones
    'output_path': 'data/transacciones_listas.csv'
}

In [20]:
# FUNCIÓN: CARGAR DATOS DESDE DIFERENTES FORMATOS


def load_data(config):
    """
    Carga datos desde un archivo según la configuración.
    Soporta CSV, Excel y JSON. Si el formato no es soportado, lanza una excepción.

    Parámetros:
        config (dict): Diccionario de configuración con las claves necesarias.

    Retorna:
        DataFrame de pandas con los datos cargados.
    """
    # Extraer parámetros de configuración
    file_path = config['file_path']
    file_type = config['file_type'].lower()  # Convertir a minúsculas para comparar

    # Verificar que el archivo existe antes de intentar cargarlo
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"El archivo {file_path} no existe.")

    # Intentar cargar según el tipo de archivo
    try:
        if file_type == 'csv':
            # Cargar CSV con opciones de configuración
            df = pd.read_csv(
                file_path,
                encoding=config.get('csv_encoding', 'utf-8'),  # Usa 'utf-8' si no se especifica
                sep=config.get('csv_sep', ',')                 # Usa ',' si no se especifica
            )
            print(f"Archivo CSV cargado correctamente: {file_path}")

        elif file_type == 'excel':
            # Cargar Excel (requiere openpyxl o xlrd)
            df = pd.read_excel(file_path)
            print(f"Archivo Excel cargado correctamente: {file_path}")

        elif file_type == 'json':
            # Cargar JSON
            df = pd.read_json(file_path)
            print(f"Archivo JSON cargado correctamente: {file_path}")

        else:
            # Si el tipo no está soportado, lanzar error
            raise ValueError(f"Tipo de archivo no soportado: {file_type}. Use 'csv', 'excel' o 'json'.")

        # Mostrar información básica del DataFrame
        print(f"Dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas.")
        print(f"Columnas encontradas: {list(df.columns)}")
        return df

    except Exception as e:
        # Capturar cualquier error durante la carga y relanzarlo con mensaje claro
        raise Exception(f"Error al cargar el archivo {file_path}: {e}")

In [23]:
df=load_data(CONFIG)

Archivo CSV cargado correctamente: C:\Repositorios\insightengine\data\test\MuestraCredito5000V2.csv
Dimensiones: 5000 filas, 6 columnas.
Columnas encontradas: ['MontoCredito', 'IngresoNeto', 'CoefCreditoAvaluo', 'MontoCuota', 'GradoAcademico', 'BuenPagador']


In [24]:
df=validate_columns(df, [CONFIG['transaction_id'], CONFIG['item_name']])

,MontoCredito,IngresoNeto,CoefCreditoAvaluo,MontoCuota,GradoAcademico,BuenPagador
0,14327,1,1,MuyBajo,Bachiller,Si
1,111404,1,1,MuyBajo,Bachiller,Si
2,21128,1,1,MuyBajo,Bachiller,Si
3,15426,2,1,MuyBajo,Bachiller,Si
4,10351,1,1,MuyBajo,Bachiller,Si
...,...,...,...,...,...,...
4995,102646,1,12,Alto,Licenciatura,No
4996,95782,1,12,Alto,Licenciatura,No
4997,51366,1,12,Alto,Licenciatura,No
4998,129504,1,12,Alto,Licenciatura,No


In [5]:
# FUNCIÓN: LIMPIEZA Y PREPROCESAMIENTO DE DATOS

def clean_data(df, config):
    """
    Aplica limpieza básica a los datos:
    - Convierte tipos de datos.
    - Elimina filas con valores nulos en columnas esenciales.
    - Elimina duplicados (misma transacción, mismo ítem).
    - Aplica filtros adicionales si están definidos.

    Parámetros:
        df (DataFrame): Datos originales.
        config (dict): Configuración con nombres de columnas y filtros.

    Retorna:
        DataFrame limpio.
    """
    # Crear una copia para no modificar el DataFrame original
    df_clean = df.copy()

    # ========== CONVERSIÓN DE TIPOS ==========

    # Convertir id_transaccion a string para evitar problemas con números mezclados con texto
    # También se asegura de que no haya valores nulos aquí (se manejarán después)
    df_clean[config['transaction_id']] = df_clean[config['transaction_id']].astype(str)

    # Convertir nombre del ítem a string y eliminar espacios en blanco al inicio y final
    df_clean[config['item_name']] = df_clean[config['item_name']].astype(str).str.strip()

    # ========== ELIMINACIÓN DE VALORES NULOS ==========

    # Registrar el número de filas antes de eliminar nulos
    before_drop = len(df_clean)

    # Eliminar filas donde id_transaccion o item_name sean nulos (NaN) o vacíos después de limpiar
    # Nota: después de convertir a string, los NaN se convierten en 'nan', por lo que también filtramos eso
    df_clean = df_clean[
        (df_clean[config['transaction_id']] != 'nan') &
        (df_clean[config['item_name']] != 'nan') &
        (df_clean[config['transaction_id']].str.strip() != '') &
        (df_clean[config['item_name']].str.strip() != '')
    ]

    after_drop = len(df_clean)
    print(f"Filas eliminadas por valores nulos o vacíos en columnas esenciales: {before_drop - after_drop}")

    # ========== ELIMINACIÓN DE DUPLICADOS ==========

    # Eliminar filas duplicadas exactas (misma transacción, mismo ítem)
    # Esto evita contar dos veces el mismo producto en una misma transacción (aunque en algunos casos podría ser válido, lo común es mantener uno)
    before_dup = len(df_clean)
    df_clean.drop_duplicates(subset=[config['transaction_id'], config['item_name']], inplace=True)
    after_dup = len(df_clean)
    print(f"Filas duplicadas eliminadas: {before_dup - after_dup}")

    # Mostrar resumen final
    print(f"Datos después de limpieza: {len(df_clean)} filas.")
    return df_clean